Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv('C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNconsumptionFinal.csv')

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(14)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [15]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 5)
Las dimensiones de testX son:  (10529, 12, 5)
Las dimensiones de valX son:  (5186, 12, 5)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1146/1146 - 41s - 36ms/step - ia: 0.5137 - loss: 0.5678 - mae: 0.6075 - rmse: 0.7410 - smape: 1.1347 - val_ia: 0.2906 - val_loss: 0.8048 - val_mae: 0.7546 - val_rmse: 0.8206 - val_smape: 1.1816

Epoch 2/128                                           

1146/1146 - 35s - 31ms/step - ia: 0.6716 - loss: 0.3614 - mae: 0.4801 - rmse: 0.5958 - smape: 0.8450 - val_ia: 0.3615 - val_loss: 0.4392 - val_mae: 0.5360 - val_rmse: 0.5984 - val_smape: 0.9135

Epoch 3/128                                           

1146/1146 - 21s - 18ms/step - ia: 0.7109 - loss: 0.2917 - mae: 0.4274 - rmse: 0.5351 - smape: 0.8021 - val_ia: 0.4030 - val_loss: 0.3451 - val_mae: 0.4800 - val_rmse: 0.5379 - val_smape: 0.9407

Epoch 4/128                                           

1146/1146 - 16s - 14ms/step - ia: 0.7325 - loss: 0.2541 - mae: 0.3975 - rmse: 0.4995 - smape: 0.7790 - val_ia: 0.4215 - val_loss: 0.2986 - val_mae: 0.4526 - val_rmse: 0.5083 - val_smape: 0.98

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

144/144 - 48s - 335ms/step - ia: 0.6059 - loss: 0.4197 - mae: 0.5009 - rmse: 0.6223 - smape: 0.9672 - val_ia: 0.6604 - val_loss: 0.3152 - val_mae: 0.4667 - val_rmse: 0.5514 - val_smape: 0.9441

Epoch 2/16                                                                         

144/144 - 26s - 182ms/step - ia: 0.8098 - loss: 0.1571 - mae: 0.3011 - rmse: 0.3948 - smape: 0.6509 - val_ia: 0.7015 - val_loss: 0.2749 - val_mae: 0.4092 - val_rmse: 0.5115 - val_smape: 0.9088

Epoch 3/16                                                                         

144/144 - 40s - 280ms/step - ia: 0.8478 - loss: 0.1054 - mae: 0.2422 - rmse: 0.3233 - smape: 0.5865 - val_ia: 0.7330 - val_loss: 0.2322 - val_mae: 0.3719 - val_rmse: 0.4451 - val_smape: 0.8109

Epoch 4/16                                                                         

144/144 - 36s - 248ms/step - ia: 0.8640 - loss: 0.0875 - mae: 0.2172 - rmse: 0

...

Epoch 1/32                                                                            

287/287 - 15s - 53ms/step - ia: 0.7776 - loss: 0.2056 - mae: 0.3320 - rmse: 0.4308 - smape: 0.7080 - val_ia: 0.7570 - val_loss: 0.1913 - val_mae: 0.3484 - val_rmse: 0.4183 - val_smape: 0.7540

Epoch 2/32                                                                            

287/287 - 8s - 29ms/step - ia: 0.8583 - loss: 0.0958 - mae: 0.2254 - rmse: 0.3081 - smape: 0.5597 - val_ia: 0.7349 - val_loss: 0.2566 - val_mae: 0.3933 - val_rmse: 0.4752 - val_smape: 0.7745

Epoch 3/32                                                                            

287/287 - 7s - 25ms/step - ia: 0.8727 - loss: 0.0786 - mae: 0.2040 - rmse: 0.2789 - smape: 0.5184 - val_ia: 0.7595 - val_loss: 0.1896 - val_mae: 0.3446 - val_rmse: 0.4025 - val_smape: 0.7275

Epoch 4/32                                                                            

287/287 - 10s - 35ms/step - ia: 0.8763 - loss: 0.0739 - mae: 0.1987 - r

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                           

1146/1146 - 29s - 25ms/step - ia: 0.1478 - loss: 0.9640 - mae: 0.8187 - rmse: 0.9776 - smape: 1.8032 - val_ia: 0.2368 - val_loss: 0.8244 - val_mae: 0.7557 - val_rmse: 0.8290 - val_smape: 1.7702

Epoch 2/128                                                                           

1146/1146 - 19s - 17ms/step - ia: 0.1896 - loss: 0.9030 - mae: 0.7953 - rmse: 0.9459 - smape: 1.6629 - val_ia: 0.2412 - val_loss: 0.7886 - val_mae: 0.7378 - val_rmse: 0.8097 - val_smape: 1.6093

Epoch 3/128                                                                           

1146/1146 - 19s - 17ms/step - ia: 0.2644 - loss: 0.8150 - mae: 0.7579 - rmse: 0.8986 - smape: 1.5090 - val_ia: 0.2484 - val_loss: 0.7560 - val_mae: 0.7207 - val_rmse: 0.7884 - val_smape: 1.4546

Epoch 4/128                                                                           

1146/1146 - 21s - 18ms/step - ia: 0.3713 - loss: 0.7002 - mae: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                            

573/573 - 9s - 16ms/step - ia: 0.3628 - loss: 1.8972 - mae: 1.1182 - rmse: 1.3724 - smape: 1.3296 - val_ia: 0.3182 - val_loss: 1.2532 - val_mae: 0.8733 - val_rmse: 1.0330 - val_smape: 1.1617

Epoch 2/16                                                                            

573/573 - 3s - 6ms/step - ia: 0.3457 - loss: 1.5285 - mae: 1.0087 - rmse: 1.2321 - smape: 1.3487 - val_ia: 0.3028 - val_loss: 1.0414 - val_mae: 0.8081 - val_rmse: 0.9513 - val_smape: 1.1898

Epoch 3/16                                                                            

573/573 - 4s - 6ms/step - ia: 0.3149 - loss: 1.3074 - mae: 0.9404 - rmse: 1.1397 - smape: 1.3871 - val_ia: 0.2884 - val_loss: 0.9161 - val_mae: 0.7703 - val_rmse: 0.9029 - val_smape: 1.2403

Epoch 4/16                                                                            

573/573 - 3s - 6ms/step - ia: 0.2778 - loss: 1.1615 - mae: 0.8925 - rmse: 

In [22]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.1, 'epochs': 2, 'layers': 4.0, 'learning_rate': 0.0001704833214115943, 'units': 3}
